# 06 — Current Score Inspection

Tujuan: melihat output score/rank **tanpa membuka outcomes**. Ini observability notebook, bukan alat override model atau rekomendasi trading.

Arahkan hanya ke score snapshot/forward artifact yang memang outcome-blind dan boleh dibaca.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SCORE_SNAPSHOT = Path(r"CHANGE_ME")
SCORE_COL = "score"
TICKER = "BBCA"

In [ ]:
if not SCORE_SNAPSHOT.exists():
    raise FileNotFoundError("Set SCORE_SNAPSHOT ke outcome-blind score parquet/CSV lokal.")

s = pd.read_parquet(SCORE_SNAPSHOT) if SCORE_SNAPSHOT.suffix.lower() == ".parquet" else pd.read_csv(SCORE_SNAPSHOT)
print("shape:", s.shape)
display(s.head())
print("columns:", list(s.columns))

## 1. Identity / duplicate sanity

In [ ]:
identity = [c for c in ("ticker", "date", "session_date") if c in s.columns]
if "ticker" not in s.columns:
    raise KeyError("Snapshot must expose ticker identity.")

if "date" in s.columns:
    dup = s.duplicated(["ticker", "date"], keep=False)
elif "session_date" in s.columns:
    dup = s.duplicated(["ticker", "session_date"], keep=False)
else:
    dup = s.duplicated(["ticker"], keep=False)
print("duplicate identity rows:", int(dup.sum()))

## 2. Top ranks/scores

In [ ]:
if SCORE_COL not in s.columns:
    likely = [c for c in s.columns if "score" in c.lower() or "rank" in c.lower()]
    raise KeyError(f"{SCORE_COL!r} not found. Likely fields: {likely}")

score = pd.to_numeric(s[SCORE_COL], errors="coerce")
view = s.assign(_score=score).sort_values("_score", ascending=False)
cols = [c for c in ["ticker", "date", "session_date", SCORE_COL, "rank", "model_id", "model_sha256"] if c in view.columns]
display(view[cols].head(20))

## 3. Inspect satu ticker

In [ ]:
row = s[s["ticker"].astype(str).str.upper().eq(TICKER)]
if row.empty:
    print(TICKER, "not present in this snapshot")
else:
    display(row.tail(1).T)

## 4. Score distribution

In [ ]:
clean = pd.to_numeric(s[SCORE_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
print(clean.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
if len(clean):
    ax = clean.hist(bins=50, figsize=(8, 3))
    ax.set_title(SCORE_COL)
    plt.show()

## Safety interpretation

Score/rank menjawab bagaimana model mengurutkan current cross-section; notebook ini **tidak** mengecek apakah prediction akhirnya benar. Jangan join ke H5/H10 outcome atau membuka vault dari sini. Jangan mengubah ranking manual hanya karena satu saham terlihat aneh—kalau ada anomaly, catat sebagai diagnosis dan audit source/features secara terpisah.